# GitLab Handbook: Dense на GPU + BM25 на CPU

Локально сначала выполните `python vector_transfer.py pack`. Загрузите полученный
`data/handbook_bm25_input.zip` в Colab Files или как Kaggle Dataset (Internet включён).
Включите GPU accelerator. Ключи RouterAI/OpenAI/Qdrant и Docker здесь не нужны.
Ноутбук считает только embeddings; результат затем импортируется в локальный Qdrant.


In [ ]:
from pathlib import Path
import zipfile

WORK = Path("/kaggle/working/handbook_job" if Path("/kaggle").exists() else "/content/handbook_job")
# При необходимости укажите свой путь к архиву.
matches = list(Path("/kaggle/input").rglob("handbook_bm25_input.zip")) if Path("/kaggle/input").exists() else [Path("/content/handbook_bm25_input.zip")]
if len(matches) != 1 or not matches[0].is_file():
    raise ValueError("Загрузите handbook_bm25_input.zip и задайте его путь в matches")
WORK.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(matches[0]) as archive:
    for name in archive.namelist():
        if not (WORK / name).resolve().is_relative_to(WORK.resolve()):
            raise ValueError("Unsafe archive path")
    archive.extractall(WORK)
print(WORK)


## Окружение
Выполните установку до импорта FastEmbed/ONNX Runtime. Если уже импортировали их —
перезапустите сессию после установки. Не устанавливайте основной requirements.txt:
он предназначен для локального CPU-окружения. Это отдельное окружение ноутбука.


In [ ]:
%pip uninstall -y fastembed onnxruntime onnxruntime-gpu fastembed-gpu
%pip install -r {WORK / "requirements-gpu.txt"}


In [ ]:
import os
import sys
import subprocess
os.chdir(WORK)
sys.path.insert(0, str(WORK))
subprocess.run(["nvidia-smi"], check=True)
import onnxruntime as ort
ort.preload_dlls(directory="")
print("ORT:", ort.__version__, "Providers:", ort.get_available_providers())
assert "CUDAExecutionProvider" in ort.get_available_providers(), "CUDA provider не найден"


## Расчёт
`LIMIT=None` — весь корпус; для проверки задайте 64 и выберите другую папку OUTPUT.
При нехватке GPU-памяти уменьшите BATCH_SIZE до 4–8. BM25 не использует GPU.
Порции по SHARD_SIZE children сохраняются атомарно. Повторный запуск продолжает
с последней сохранённой порции; параметры батча можно менять.

Для Colab можно смонтировать Google Drive и направить OUTPUT туда. В Kaggle файлы
в /kaggle/working нужно сохранить как output версии или скачать до потери сессии.
Для продолжения после полного сброса восстановите всю папку OUTPUT из сохранённой
копии. GPU сам по себе не делает временный диск постоянным.


In [ ]:
from vector_transfer import export_vectors
from hybrid_search import Embedders

OUTPUT = WORK / "handbook_vectors_bm25"  # Можно указать папку на Google Drive
LIMIT = None
BATCH_SIZE = 32
SPARSE_BATCH_SIZE = 128
SHARD_SIZE = 256
REUSE_DENSE = None  # Например Path("/content/handbook_vectors") со старым SPLADE-экспортом

encoder = Embedders(cache=WORK / "model_cache", device="cuda", sparse_batch_size=SPARSE_BATCH_SIZE)
print("Dense:", encoder.dense.model.model.get_providers())
print("BM25 CPU:", encoder.sparse.settings)
manifest = export_vectors(WORK / "snapshot", OUTPUT, encoder,
                         batch_size=BATCH_SIZE, shard_size=SHARD_SIZE, limit=LIMIT,
                         resume=(OUTPUT / "manifest.json").exists(), reuse_dense=REUSE_DENSE)
print(manifest["status"], manifest["indexed_children"])


## Скачивание результата
Архив содержит векторы, документы, parents и параметры моделей. Не требует
повторного вычисления embeddings при импорте. Если архив большой, скачайте его
через файловый браузер Colab или Kaggle Output.


In [ ]:
from vector_transfer import pack_result
from IPython.display import FileLink, display

RESULT_ZIP = WORK / "handbook_vectors_bm25.zip"
pack_result(OUTPUT, RESULT_ZIP)
print(RESULT_ZIP, round(RESULT_ZIP.stat().st_size / 1024**2), "MiB")
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(str(RESULT_ZIP))
else:
    display(FileLink(str(RESULT_ZIP)))


## На своём компьютере
Распакуйте архив в `data/handbook_vectors_bm25`, затем из папки проекта:
```powershell
docker compose up -d
.\.venv\Scripts\python.exe vector_transfer.py import --input data/handbook_vectors_bm25
.\.venv\Scripts\python.exe hybrid_search.py search "How do I request time off?" --alias handbook
```
Если LIMIT был задан, импорт создаст alias `handbook_sample` вместо полного handbook.
После импорта запросы можно считать на CPU с теми же моделями. Новые вопросы
требуют embeddings только для вопроса, корпус повторно не кодируется.
